In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive")

# Change this after checking your Drive structure
PROCESSED_DIR = BASE_DIR / "DAIC_WOZ_VIDEO_BRANCH" / "data" / "processed"

print("Exists:", PROCESSED_DIR.exists())
print("Path:", PROCESSED_DIR)

Exists: True
Path: /content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/data/processed


In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

video_dataset = pd.read_csv(PROCESSED_DIR / "video_dataset_with_labels.csv")

target_col = "PHQ_Binary"

drop_cols = ["Participant_ID", "split", "PHQ_Binary", "PHQ_Score"]
feature_cols = [c for c in video_dataset.columns if c not in drop_cols]

# train_split = video_dataset
train_df = video_dataset[video_dataset["split"] == "train"][0:100].copy()
dev_df = video_dataset[video_dataset["split"] == "train"][100:164].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_dev = dev_df[feature_cols]
y_dev = dev_df[target_col]

print(X_train.shape, X_dev.shape)
print(y_train.value_counts())
print(y_dev.value_counts())

(100, 362) (63, 362)
PHQ_Binary
0    71
1    29
Name: count, dtype: int64
PHQ_Binary
0    55
1     8
Name: count, dtype: int64


In [4]:
models = {
    "logistic_regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),
    "svm_rbf": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced"))
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        ))
    ]),
    "gradient_boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", GradientBoostingClassifier(random_state=42))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_dev)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_dev)[:, 1]
    else:
        y_prob = None

    bal_acc = balanced_accuracy_score(y_dev, y_pred)

    try:
        auc = roc_auc_score(y_dev, y_prob) if y_prob is not None else np.nan
    except:
        auc = np.nan

    print("\n==============================")
    print(name)
    print("==============================")
    print("Balanced Accuracy:", bal_acc)
    print("ROC-AUC:", auc)
    print(confusion_matrix(y_dev, y_pred))
    print(classification_report(y_dev, y_pred))

    results.append({
        "model": name,
        "balanced_accuracy": bal_acc,
        "roc_auc": auc
    })

results_df = pd.DataFrame(results)
results_df


logistic_regression
Balanced Accuracy: 0.4602272727272727
ROC-AUC: 0.47500000000000003
[[30 25]
 [ 5  3]]
              precision    recall  f1-score   support

           0       0.86      0.55      0.67        55
           1       0.11      0.38      0.17         8

    accuracy                           0.52        63
   macro avg       0.48      0.46      0.42        63
weighted avg       0.76      0.52      0.60        63


svm_rbf
Balanced Accuracy: 0.36363636363636365
ROC-AUC: 0.7522727272727272
[[40 15]
 [ 8  0]]
              precision    recall  f1-score   support

           0       0.83      0.73      0.78        55
           1       0.00      0.00      0.00         8

    accuracy                           0.63        63
   macro avg       0.42      0.36      0.39        63
weighted avg       0.73      0.63      0.68        63


random_forest
Balanced Accuracy: 0.4636363636363636
ROC-AUC: 0.32613636363636367
[[51  4]
 [ 8  0]]
              precision    recall  f1-score

,model,balanced_accuracy,roc_auc
0,logistic_regression,0.460227,0.475000
1,svm_rbf,0.363636,0.752273
2,random_forest,0.463636,0.326136
3,gradient_boosting,0.389773,0.293182


In [5]:
RESULTS_DIR = BASE_DIR / "DAIC_WOZ_VIDEO_BRANCH" / "results" / "tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(RESULTS_DIR / "video_model_results.csv", index=False)